
# Optimized BioCatch Vishing Data Augmentation Pipeline

Versión optimizada para ejecución en Kaggle P100/T4 con enfoque en:

- reducción de consumo RAM
- generación chunked
- exportación incremental
- entrenamiento progresivo
- control de memoria
- parquet streaming
- validaciones eficientes

Objetivo:
- generar ~1M observaciones
- mantener fraude ~1%
- evitar OOM en Kaggle


In [ ]:
# ====================================
# FIX: PyTorch compatible con P100 (sm_60 / Pascal)
# ====================================
# Kaggle 2025 ships torch 2.10+cu128 which dropped sm_60 (P100/Pascal).
# Estrategia:
#   1. Forzar reinstalación de torch 2.1.2+cu118 (último con sm_60)
#   2. Verificar con smoke-test que el kernel CUDA funciona
#   3. Si falla, parchear ctgan para forzar CPU (fallback seguro)
#
# DESPUÉS DE EJECUTAR ESTA CELDA:
#   Session > Restart & Run All  (obligatorio, sin esto Python usa el .so viejo)
# ====================================

import subprocess, sys, os

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out[-3000:])
    return r.returncode

print("=== PASO 1: Desinstalar torch actual ===")
run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"])

print("\n=== PASO 2: Instalar torch 2.1.2+cu118 (incluye sm_60 para P100) ===")
rc = run([
    sys.executable, "-m", "pip", "install",
    "torch==2.1.2+cu118",
    "torchvision==0.16.2+cu118",
    "--index-url", "https://download.pytorch.org/whl/cu118",
    "--no-cache-dir",
])
print(f"\npip returncode: {rc}")

print("\n=== PASO 3: Verificar ubicación del .so instalado ===")
import importlib.util
spec = importlib.util.find_spec("torch")
print(f"torch encontrado en: {spec.origin if spec else 'NO ENCONTRADO'}")

print("\n=== PASO 4: Smoke-test CUDA con torch recién instalado ===")
# Necesitamos reimportar desde el path correcto sin reiniciar kernel
# (limitado — la verificación real ocurre después del restart)
import importlib
if 'torch' in sys.modules:
    # Forzar reimport
    for mod in list(sys.modules.keys()):
        if mod == 'torch' or mod.startswith('torch.'):
            del sys.modules[mod]

try:
    import torch
    print(f"torch.__version__ = {torch.__version__}")
    print(f"torch.__file__    = {torch.__file__}")
    if torch.cuda.is_available():
        t = torch.zeros(4, 4).cuda()
        _ = t @ t
        del t
        torch.cuda.empty_cache()
        print(f"GPU smoke-test: OK ✓  ({torch.cuda.get_device_name(0)})")
        print("\n✅ Listo. Haz Session > Restart & Run All para continuar.")
    else:
        print("CUDA no detectado. Verifica que el acelerador esté activo en Settings > Accelerator.")
except Exception as e:
    print(f"Error en smoke-test: {e}")
    print("\n⚠️  Haz Session > Restart & Run All de todas formas.")
    print("Si el error persiste tras el restart, los sintetizadores usarán CPU (fallback automático).")


In [1]:

# ====================================
# INSTALLS (KAGGLE)
# ====================================
# sdv 1.36 requiere ctgan >= 0.10; ctgan 0.10+ funciona con torch 2.1
!pip install -q sdv pyarrow fastparquet psutil


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.8/204.8 kB 4.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 31.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.8 MB/s eta 0:00:00:00:01


In [2]:

# ====================================
# IMPORTS
# ====================================

import warnings
warnings.filterwarnings("ignore")

import gc
import os
import random
import psutil

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

from sdv.single_table import CTGANSynthesizer
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

from tqdm import tqdm

import pyarrow.parquet as pq
import pyarrow as pa
import torch


In [3]:

# ====================================
# CONFIG
# ====================================

DATASET_PATH = "/kaggle/input/datasets/jm0ntyy/dataset-sintetico-biocatch-vishing-overlap/dataset_sintetico_biocatch_vishing_overlap.csv"

OUTPUT_PARQUET = "/kaggle/working/biocatch_augmented_1M.parquet"

TARGET_ROWS = 1_000_000
TARGET_VISHING = 10_000
TARGET_LEGIT = TARGET_ROWS - TARGET_VISHING

CHUNK_SIZE = 100_000

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ====================================
# GPU / PYTORCH ENV SETUP
# ====================================

# Limitar fragmentación de memoria en la GPU
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"

# Forzar que CUDA sea visible (Kaggle debería tenerlo ya, pero por si acaso)
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # descomentar solo si hay problemas

# ====================================
# MEMORY HELPERS
# ====================================

def ram_free_gb():
    return psutil.virtual_memory().available / 1024**3

def vram_free_gb():
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info(0)
        return free / 1024**3
    return 0.0

def print_resources(label=""):
    prefix = f"[{label}] " if label else ""
    ram = psutil.virtual_memory()
    msg = (
        f"{prefix}RAM: {ram.used/1024**3:.1f}/{ram.total/1024**3:.1f} GB "
        f"({ram.percent:.0f}% usado)"
    )
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info(0)
        msg += f" | VRAM libre: {free/1024**3:.2f}/{total/1024**3:.2f} GB"
    print(msg)

def aggressive_gc():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

RAM_LOW_THRESHOLD_GB = 2.0   # pausar si RAM libre cae por debajo de esto
VRAM_LOW_THRESHOLD_GB = 1.5  # advertencia si VRAM libre cae por debajo de esto

print("Config cargada.")
print_resources("INICIO")


In [4]:

# ====================================
# LOAD DATASET
# ====================================

df = pd.read_csv(DATASET_PATH)

print(df.shape)

df.head()


(50000, 61)


,session_id,customer_id,session_timestamp,device_type,os_type,app_version,avg_keyhold_ms,avg_interkey_latency_ms,typing_speed_cps,keystroke_variability,...,biocatch_genuine_score,biocatch_ato_indicator,biocatch_social_eng_indicator,biocatch_bot_indicator,errors_per_minute,interactions_per_s,hesitation_composite,is_vishing,days_to_claim,claim_category
0,SES-NO-000001,CUS-53613,2024-08-24 14:54:24,mobile,iOS,v13.0,97.7,178.4,3.777,0.240,...,896,0,0,0,8.717,2.4,1.1518,0,-1,none
1,SES-NO-000002,CUS-43723,2025-02-06 21:28:41,mobile,iOS,v12.3,72.8,78.5,3.016,0.086,...,670,1,0,0,8.717,2.4,1.0316,0,-1,none
2,SES-NO-000003,CUS-93608,2025-05-15 15:22:09,web,Windows,v12.3,86.7,152.3,4.776,0.141,...,838,0,0,0,0.000,2.4,1.1518,0,-1,none
3,SES-NO-000004,CUS-25829,2024-08-08 06:00:27,mobile,Android,v13.0,82.9,176.7,4.992,0.171,...,570,0,0,0,8.717,2.4,1.1518,0,-1,none
4,SES-NO-000005,CUS-46701,2024-09-04 11:09:02,mobile,Android,v12.1,69.9,204.0,6.337,0.133,...,1000,0,0,0,8.717,2.4,1.1518,0,-1,none


In [5]:

# ====================================
# MEMORY OPTIMIZATION
# ====================================

def optimize_memory(df):

    for col in df.columns:

        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')

        elif df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')

        elif df[col].dtype == 'object':

            nunique = df[col].nunique()

            if nunique / len(df) < 0.5:
                df[col] = df[col].astype('category')

    return df

df = optimize_memory(df)

df.info(memory_usage='deep')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 61 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   session_id                     50000 non-null  object  
 1   customer_id                    50000 non-null  category
 2   session_timestamp              50000 non-null  object  
 3   device_type                    50000 non-null  category
 4   os_type                        50000 non-null  category
 5   app_version                    50000 non-null  category
 6   avg_keyhold_ms                 50000 non-null  float32 
 7   avg_interkey_latency_ms        50000 non-null  float32 
 8   typing_speed_cps               50000 non-null  float32 
 9   keystroke_variability          50000 non-null  float32 
 10  segmented_typing_ratio         50000 non-null  float32 
 11  avg_touch_pressure             50000 non-null  float32 
 12  avg_touch_size_px              5

In [6]:

# ====================================
# REMOVE LEAKAGE
# ====================================

leakage_cols = [
    'biocatch_risk_score',
    'biocatch_genuine_score',
    'biocatch_ato_indicator',
    'biocatch_social_eng_indicator',
    'biocatch_bot_indicator',
    'claim_category',
    'days_to_claim'
]

existing_leakage = [
    c for c in leakage_cols if c in df.columns
]

df_model = df.drop(columns=existing_leakage)

print(df_model.shape)


(50000, 54)


In [7]:

# ====================================
# SPLIT CLASSES
# ====================================

legit_df = df_model[df_model['is_vishing'] == 0].copy()
vishing_df = df_model[df_model['is_vishing'] == 1].copy()

print(legit_df.shape)
print(vishing_df.shape)


(47500, 54)
(2500, 54)


In [9]:

# ====================================
# SDV DATA PREPARATION
# ====================================

def prepare_for_sdv(df_input):

    df_sdv = df_input.copy()

    category_cols = df_sdv.select_dtypes(include=['category']).columns

    for col in category_cols:
        df_sdv[col] = df_sdv[col].astype('object')

    return df_sdv



# Training Strategy

Primero validar pipeline con pocos epochs.
Luego aumentar progresivamente.

Esto evita perder horas por:
- OOM
- errores
- configuraciones inválidas


In [10]:

# ====================================
# SDV METADATA
# ====================================

metadata = SingleTableMetadata()

metadata.detect_from_dataframe(df_model)

metadata


{
    "METADATA_SPEC_VERSION": "SINGLE_TABLE_V1",
    "primary_key": "session_id",
    "columns": {
        "session_id": {
            "sdtype": "id"
        },
        "customer_id": {
            "sdtype": "id"
        },
        "session_timestamp": {
            "datetime_format": "%Y-%m-%d %H:%M:%S",
            "sdtype": "datetime"
        },
        "device_type": {
            "sdtype": "categorical"
        },
        "os_type": {
            "sdtype": "categorical"
        },
        "app_version": {
            "sdtype": "categorical"
        },
        "avg_keyhold_ms": {
            "sdtype": "numerical"
        },
        "avg_interkey_latency_ms": {
            "sdtype": "numerical"
        },
        "typing_speed_cps": {
            "sdtype": "numerical"
        },
        "keystroke_variability": {
            "sdtype": "numerical"
        },
        "segmented_typing_ratio": {
            "sdtype": "numerical"
        },
        "avg_touch_pressure": {
            "

In [11]:

# ====================================
# FAST CTGAN CONFIG
# ====================================

import torch

# --- Verificación de arquitectura (P100 = sm_60) ---
USE_GPU = False
_gpu_device = None

if torch.cuda.is_available():
    try:
        _t = torch.zeros(4, 4).cuda()
        _ = _t @ _t
        del _t
        torch.cuda.empty_cache()
        USE_GPU = True
        _gpu_device = "cuda"
        name = torch.cuda.get_device_name(0)
        free, total = torch.cuda.mem_get_info(0)
        print(f"GPU smoke-test: OK ✓")
        print(f"  Dispositivo : {name}")
        print(f"  VRAM        : {total/1024**3:.1f} GB total | {free/1024**3:.2f} GB libre")
        print(f"  torch       : {torch.__version__}")
    except Exception as e:
        print(f"⚠️  GPU smoke-test FALLÓ: {e}")
        print("   → Entrenamiento en CPU (más lento pero funcional)")
        print("   → Verifica que ejecutaste la celda FIX y reiniciaste el kernel.")
        USE_GPU = False
        _gpu_device = None
else:
    print("⚠️  CUDA no disponible. Entrenamiento en CPU.")
    print("   Verifica que el acelerador P100 esté activo en Settings > Accelerator")

print(f"\nUSE_GPU = {USE_GPU}")

# batch_size debe ser múltiplo de pac (por defecto pac=10)
_batch = 500

ctgan_legit = CTGANSynthesizer(
    metadata=metadata,
    epochs=20,
    batch_size=_batch,
    discriminator_steps=1,
    verbose=True,
    enable_gpu=USE_GPU,
    cuda=_gpu_device,   # None → CPU, "cuda" → GPU
)

ctgan_vishing = CTGANSynthesizer(
    metadata=metadata,
    epochs=20,
    batch_size=_batch,
    discriminator_steps=1,
    verbose=True,
    enable_gpu=USE_GPU,
    cuda=_gpu_device,
)

print_resources("POST-INIT CTGAN")


In [12]:
# ====================================
# DIAGNOSTIC: Environment & Synthesizers
# ====================================

import inspect
import sdv

print('sdv.__version__ =', getattr(sdv, '__version__', 'unknown'))
print('torch.__version__ =', torch.__version__)

print('\nConstructor signatures:')
print('CTGANSynthesizer', inspect.signature(CTGANSynthesizer))
print('TVAESynthesizer', inspect.signature(TVAESynthesizer))

print('\nGlobal synth objects available:')
for name in ['ctgan_legit', 'ctgan_vishing', 'tvae']:
    obj = globals().get(name)
    print(f"- {name}: {'present' if obj is not None else 'MISSING'}")

print('\nSynthesizer GPU-related attrs (if present):')
def dump_attrs(obj, obj_name):
    if obj is None:
        return
    for a in ['enable_gpu', 'cuda', 'device']:
        if hasattr(obj, a):
            try:
                val = getattr(obj, a)
                flag = " ✓ GPU" if a == 'enable_gpu' and val is True else (" ✗ CPU" if a == 'enable_gpu' else "")
                print(f"  {obj_name}.{a} = {val}{flag}")
            except Exception as e:
                print(f"  {obj_name}.{a} -> error: {e}")

for name in ['ctgan_legit', 'ctgan_vishing', 'tvae']:
    obj = globals().get(name)
    if obj is not None:
        dump_attrs(obj, name)

print('\nTorch / CUDA info:')
try:
    avail = torch.cuda.is_available()
    print('torch.cuda.is_available() =', avail)
    print('torch.cuda.device_count() =', torch.cuda.device_count())
    if avail:
        print('torch.cuda.get_device_name(0) =', torch.cuda.get_device_name(0))
        free, total = torch.cuda.mem_get_info(0)
        print(f'VRAM: {total/1024**3:.1f} GB total, {free/1024**3:.2f} GB libre')
except Exception as e:
    print('CUDA query error:', e)

print('\nEnv vars:')
print('CUDA_VISIBLE_DEVICES   =', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('CUDA_LAUNCH_BLOCKING   =', os.environ.get('CUDA_LAUNCH_BLOCKING'))
print('PYTORCH_CUDA_ALLOC_CONF=', os.environ.get('PYTORCH_CUDA_ALLOC_CONF'))

print('\nPython executable:', sys.executable)
print('Platform:', sys.platform)

print('\nRecursos del sistema:')
print_resources("DIAGNOSTIC")

# Smoke-test GPU con tensor pequeño
if USE_GPU:
    try:
        t = torch.randn(100, 100).cuda()
        _ = t @ t.T
        del t
        torch.cuda.empty_cache()
        print('\nSmoke-test GPU: OK (100x100 matmul en CUDA)')
    except Exception as e:
        print(f'\nSmoke-test GPU FALLÓ: {e}')
        print('Revisa que el acelerador P100 esté habilitado en Settings > Accelerator')
else:
    print('\n[ADVERTENCIA] CUDA no disponible. Verifica Settings > Accelerator en Kaggle.')


sdv.__version__ = 1.36.2
torch.__version__ = 2.10.0+cu128

Constructor signatures:
CTGANSynthesizer (metadata, enforce_min_max_values=True, enforce_rounding=True, locales=['en_US'], embedding_dim=128, generator_dim=(256, 256), discriminator_dim=(256, 256), generator_lr=0.0002, generator_decay=1e-06, discriminator_lr=0.0002, discriminator_decay=1e-06, batch_size=500, discriminator_steps=1, log_frequency=True, verbose=False, epochs=300, pac=10, enable_gpu=True, cuda=None)
TVAESynthesizer (metadata, enforce_min_max_values=True, enforce_rounding=True, embedding_dim=128, compress_dims=(128, 128), decompress_dims=(128, 128), l2scale=1e-05, batch_size=500, verbose=False, epochs=300, loss_factor=2, enable_gpu=True, cuda=None)

Global synth objects available:
- ctgan_legit: present
- ctgan_vishing: present
- tvae: MISSING

Synthesizer GPU-related attrs (if present):
ctgan_legit.enable_gpu = False
ctgan_vishing.enable_gpu = False

Torch / CUDA info:
torch.cuda.is_available() = True
torch.cuda.de

In [ ]:

# ====================================
# TRAIN LEGIT CTGAN
# ====================================

print_resources("PRE-TRAIN LEGIT")

legit_df_sdv = prepare_for_sdv(legit_df)
legit_df_sdv_train = legit_df_sdv.sample(
    n=min(50000, len(legit_df_sdv)),
    random_state=RANDOM_STATE
).copy()

print(f"Entrenando CTGAN legit con {len(legit_df_sdv_train):,} filas...")
print(legit_df_sdv_train.shape)

ctgan_legit.fit(legit_df_sdv_train)

aggressive_gc()
print_resources("POST-TRAIN LEGIT")
print("Legit CTGAN trained ✓")


In [ ]:

# ====================================
# TRAIN VISHING CTGAN
# ====================================

print_resources("PRE-TRAIN VISHING")

vishing_df_sdv = prepare_for_sdv(vishing_df)
print(f"Entrenando CTGAN vishing con {len(vishing_df_sdv):,} filas...")

ctgan_vishing.fit(vishing_df_sdv)

aggressive_gc()
print_resources("POST-TRAIN VISHING")
print("Vishing CTGAN trained ✓")


In [ ]:

# ====================================
# TVAE  (GPU enabled)
# ====================================

print_resources("PRE-TRAIN TVAE")

df_model_sdv = prepare_for_sdv(df_model)

tvae = TVAESynthesizer(
    metadata=metadata,
    epochs=15,
    batch_size=512,
    verbose=True,
    enable_gpu=USE_GPU,
    cuda=_gpu_device,
)

print(f"Entrenando TVAE con {len(df_model_sdv):,} filas...")
tvae.fit(df_model_sdv)

aggressive_gc()
print_resources("POST-TRAIN TVAE")
print("TVAE trained ✓")


In [ ]:

# ====================================
# CONSTRAINT ENGINE
# ====================================

def apply_constraints(df_aug):

    if 'phone_call_active' in df_aug.columns and 'call_overlap_duration_s' in df_aug.columns:

        mask = df_aug['phone_call_active'] == 0

        df_aug.loc[mask, 'call_overlap_duration_s'] = 0

    if 'transaction_attempted' in df_aug.columns and 'transaction_amount_cop' in df_aug.columns:

        mask = df_aug['transaction_attempted'] == 0

        df_aug.loc[mask, 'transaction_amount_cop'] = 0

    if 'dead_time_ratio' in df_aug.columns:

        df_aug['dead_time_ratio'] = np.clip(
            df_aug['dead_time_ratio'],
            0,
            1
        )

    if 'segmented_typing_ratio' in df_aug.columns:

        df_aug['segmented_typing_ratio'] = np.clip(
            df_aug['segmented_typing_ratio'],
            0,
            1
        )

    return df_aug


In [ ]:

# ====================================
# HARD CASE GENERATION
# ====================================

def generate_hard_legit(df_input, n):

    hard = df_input.sample(
        n=n,
        replace=True
    ).copy()

    if 'phone_call_active' in hard.columns:
        hard['phone_call_active'] = np.random.binomial(
            1,
            0.35,
            size=n
        )

    if 'hesitation_count' in hard.columns:
        hard['hesitation_count'] *= np.random.uniform(
            1.2,
            1.8,
            size=n
        )

    hard['is_vishing'] = 0

    return hard

def generate_soft_vishing(df_input, n):

    soft = df_input.sample(
        n=n,
        replace=True
    ).copy()

    if 'hesitation_count' in soft.columns:
        soft['hesitation_count'] *= np.random.uniform(
            0.5,
            0.8,
            size=n
        )

    soft['is_vishing'] = 1

    return soft


In [ ]:

# ====================================
# PARQUET STREAMING WRITER
# ====================================

parquet_writer = None

def append_parquet(df_chunk):

    global parquet_writer

    table = pa.Table.from_pandas(df_chunk)

    if parquet_writer is None:

        parquet_writer = pq.ParquetWriter(
            OUTPUT_PARQUET,
            table.schema,
            compression='snappy'
        )

    parquet_writer.write_table(table)



# Chunked Generation

La generación chunked evita:
- RAM overflow
- copies gigantes
- crashes de Kaggle

Cada chunk:
- se genera
- se valida
- se exporta
- se libera de memoria


In [ ]:

# ====================================
# GENERATE LEGIT CHUNKS
# ====================================

print_resources("INICIO GENERACIÓN LEGIT")
remaining_legit = TARGET_LEGIT
chunk_num = 0

while remaining_legit > 0:

    # Guardia de memoria RAM
    if ram_free_gb() < RAM_LOW_THRESHOLD_GB:
        print(f"[AVISO] RAM libre crítica ({ram_free_gb():.2f} GB). Esperando GC...")
        aggressive_gc()
        import time; time.sleep(3)
        if ram_free_gb() < RAM_LOW_THRESHOLD_GB:
            print("[ERROR] RAM insuficiente. Deteniendo generación para evitar OOM.")
            break

    # Advertencia de VRAM
    if USE_GPU and vram_free_gb() < VRAM_LOW_THRESHOLD_GB:
        print(f"[AVISO] VRAM libre baja ({vram_free_gb():.2f} GB). Vaciando caché CUDA...")
        aggressive_gc()

    current_chunk = min(CHUNK_SIZE, remaining_legit)
    chunk_num += 1

    ctgan_chunk = ctgan_legit.sample(num_rows=int(current_chunk * 0.7))
    tvae_chunk = tvae.sample(num_rows=int(current_chunk * 0.2))
    bootstrap_chunk = legit_df.sample(n=int(current_chunk * 0.1), replace=True).copy()
    hard_legit = generate_hard_legit(legit_df, n=max(1000, int(current_chunk * 0.02)))

    chunk = pd.concat([ctgan_chunk, tvae_chunk, bootstrap_chunk, hard_legit], ignore_index=True)
    chunk['is_vishing'] = 0
    chunk = apply_constraints(chunk)
    chunk = optimize_memory(chunk)

    append_parquet(chunk)

    remaining_legit -= len(chunk)
    print(f"[Chunk {chunk_num}] Remaining legit: {remaining_legit:,}", end="  ")
    print_resources()

    del ctgan_chunk, tvae_chunk, bootstrap_chunk, hard_legit, chunk
    aggressive_gc()

print("\nGeneración legit completa.")


In [ ]:

# ====================================
# GENERATE VISHING CHUNKS
# ====================================

print_resources("INICIO GENERACIÓN VISHING")
remaining_vishing = TARGET_VISHING
chunk_num = 0

while remaining_vishing > 0:

    if ram_free_gb() < RAM_LOW_THRESHOLD_GB:
        print(f"[AVISO] RAM libre crítica ({ram_free_gb():.2f} GB). Esperando GC...")
        aggressive_gc()
        import time; time.sleep(3)
        if ram_free_gb() < RAM_LOW_THRESHOLD_GB:
            print("[ERROR] RAM insuficiente. Deteniendo generación para evitar OOM.")
            break

    if USE_GPU and vram_free_gb() < VRAM_LOW_THRESHOLD_GB:
        print(f"[AVISO] VRAM libre baja ({vram_free_gb():.2f} GB). Vaciando caché CUDA...")
        aggressive_gc()

    current_chunk = min(20_000, remaining_vishing)
    chunk_num += 1

    ctgan_chunk = ctgan_vishing.sample(num_rows=int(current_chunk * 0.6))
    tvae_chunk = tvae.sample(num_rows=int(current_chunk * 0.2))
    bootstrap_chunk = vishing_df.sample(n=int(current_chunk * 0.2), replace=True).copy()
    soft_vishing = generate_soft_vishing(vishing_df, n=max(500, int(current_chunk * 0.05)))

    chunk = pd.concat([ctgan_chunk, tvae_chunk, bootstrap_chunk, soft_vishing], ignore_index=True)
    chunk['is_vishing'] = 1
    chunk = apply_constraints(chunk)
    chunk = optimize_memory(chunk)

    append_parquet(chunk)

    remaining_vishing -= len(chunk)
    print(f"[Chunk {chunk_num}] Remaining vishing: {remaining_vishing:,}", end="  ")
    print_resources()

    del ctgan_chunk, tvae_chunk, bootstrap_chunk, soft_vishing, chunk
    aggressive_gc()

print("\nGeneración vishing completa.")


In [ ]:

# ====================================
# CLOSE WRITER
# ====================================

if parquet_writer:
    parquet_writer.close()

print("Parquet export completed")


In [ ]:

# ====================================
# LOAD FINAL DATASET
# ====================================

final_df = pd.read_parquet(OUTPUT_PARQUET)

print(final_df.shape)

final_df.head()


In [ ]:

# ====================================
# ADVERSARIAL VALIDATION
# ====================================

orig = df_model.sample(
    n=min(50000, len(df_model)),
    random_state=42
).copy()

orig['synthetic'] = 0

synth = final_df.sample(
    n=min(150000, len(final_df)),
    random_state=42
).copy()

synth['synthetic'] = 1

adv_df = pd.concat([orig, synth])

drop_cols = [
    c for c in [
        'session_id',
        'customer_id',
        'session_timestamp'
    ]
    if c in adv_df.columns
]

X = adv_df.drop(columns=['synthetic'] + drop_cols)

for col in X.select_dtypes(include=['object', 'category']).columns:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

X = X.fillna(0)

y = adv_df['synthetic']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

clf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

clf.fit(X_train, y_train)

preds = clf.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, preds)

print(f"Adversarial Validation AUC: {auc:.4f}")


In [ ]:

# ====================================
# FINAL STATS
# ====================================

print(final_df['is_vishing'].value_counts())

print(final_df['is_vishing'].value_counts(normalize=True))

print(final_df.memory_usage(deep=True).sum() / 1024**2, "MB")



# Recomendaciones de Ejecución

## Primera corrida

NO generar 1M inicialmente.

Usar:

```python
TARGET_ROWS = 100_000
```

para validar.

---

## Luego

Escalar progresivamente:

- 250k
- 500k
- 1M

---

## Configuración recomendada Kaggle

- GPU P100/T4
- Internet OFF
- RAM High

---

## Tiempo esperado

| Etapa | Tiempo |
|---|---|
| Training | 30-60 min |
| Generation | 10-30 min |
| Validation | 5-15 min |

Tiempo total esperado:

~45 min a 1.5h
